In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

import tslab.config as cfg
from tslab.data import load_market_data
from tslab.features import make_dataset, check_no_lookahead
from tslab.validation import walk_forward_splits, walk_forward_predict, describe_splits
from tslab.metrics import compare_models, out_of_sample_r2, directional_accuracy

print(cfg.TARGET_TICKER, cfg.INITIAL_TRAIN, cfg.TEST_SIZE)   # expect: MSFT 1000 63

MSFT 1000 63


In [2]:
frame = load_market_data()
check_no_lookahead(frame)                     # must return True

dataset = make_dataset(frame, horizon=1)
dataset = dataset.dropna(subset=["target_return"])   # drop the final row that has no target yet

feature_cols = [c for c in dataset.columns if c not in ("close", "target_return", "target_close")]
X = dataset[feature_cols]
y = dataset["target_return"]
print(len(dataset), "rows,", len(feature_cols), "features")
print(feature_cols)

Loaded 2981 trading days of MSFT, 2015-01-02 to 2026-11-09, with 7 predictor series.
2781 rows, 47 features
['ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag4', 'ret_lag5', 'ret_5d', 'ret_10d', 'ret_21d', 'ret_63d', 'vol_5d', 'vol_21d', 'vol_63d', 'vol_ratio', 'range_hl', 'atr_14', 'rsi_14', 'macd_hist', 'bb_percent_b', 'bb_width', 'dist_sma_20', 'dist_sma_50', 'dist_sma_200', 'gap_overnight', 'move_intraday', 'volume_rel_20d', 'SP500_ret_1d', 'SP500_ret_5d', 'TECH_ret_1d', 'TECH_ret_5d', 'GOLD_ret_1d', 'GOLD_ret_5d', 'BRENT_ret_1d', 'BRENT_ret_5d', 'USD_ret_1d', 'USD_ret_5d', 'VIX_level', 'VIX_change_1d', 'US10Y_level', 'US10Y_change_5d', 'relative_to_SP500_5d', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'month_sin', 'month_cos']


In [15]:
splits = list(walk_forward_splits(len(dataset), cfg.INITIAL_TRAIN, cfg.TEST_SIZE, gap=0))
print("n splits:", len(splits))
describe_splits(splits, dataset.index)

n splits: 29


,train_rows,train_from,train_to,test_rows,test_from,test_to
fold,,,,,,
1,1000,2015-10-16,2019-10-07,63,2019-10-08,2020-01-07
2,1063,2015-10-16,2020-01-07,63,2020-01-08,2020-04-07
3,1126,2015-10-16,2020-04-07,63,2020-04-08,2020-07-08
4,1189,2015-10-16,2020-07-08,63,2020-07-09,2020-10-06
5,1252,2015-10-16,2020-10-06,63,2020-10-07,2021-01-06
6,1315,2015-10-16,2021-01-06,63,2021-01-07,2021-04-08
7,1378,2015-10-16,2021-04-08,63,2021-04-09,2021-07-08
8,1441,2015-10-16,2021-07-08,63,2021-07-09,2021-10-06
9,1504,2015-10-16,2021-10-06,63,2021-10-07,2022-01-05


In [16]:
ridge_pred = walk_forward_predict(lambda: Ridge(alpha=10.0), X, y, splits, standardize=True)
print(ridge_pred.notna().sum(), "out-of-sample predictions")

1781 out-of-sample predictions


In [17]:
results = compare_models(dataset, {"Ridge": ridge_pred})
results

,RMSE ($),MAE ($),MAPE (%),Directional accuracy,Out-of-sample R2 (returns)
Naive (no change),5.788400,3.993950,1.301695,NaN,0.00000
Ridge,5.994586,4.175962,1.360648,0.507019,-0.05623


In [18]:
mask = ridge_pred.notna()
actual = dataset.loc[mask, "target_return"]
pred = ridge_pred[mask]

print("Out-of-sample R2 vs naive:", out_of_sample_r2(actual, pred))
print("Directional accuracy:", directional_accuracy(actual, pred))

Out-of-sample R2 vs naive: -0.05623015453775282
Directional accuracy: 0.5070185289163391


In [19]:
out = dataset[["close", "target_close", "target_return"]].copy()
out["ridge_pred"] = ridge_pred
out["naive_pred"] = 0.0
out.dropna(subset=["ridge_pred"]).to_csv(cfg.OUTPUT_DIR / "member2_ridge_predictions.csv")
pd.DataFrame(results).to_csv(cfg.OUTPUT_DIR / "member2_metrics.csv")

In [20]:
print("dataset rows:", len(dataset))
print("ridge_pred length:", len(ridge_pred))
print("ridge_pred non-null:", ridge_pred.notna().sum())
print("ridge_pred index head:", ridge_pred.index[:5].tolist())
print("dataset index head:", dataset.index[:5].tolist())
print("indexes equal:", ridge_pred.index.equals(dataset.index))
print("target_return non-null in dataset:", dataset["target_return"].notna().sum())

dataset rows: 2781
ridge_pred length: 2781
ridge_pred non-null: 1781
ridge_pred index head: [Timestamp('2015-10-16 00:00:00'), Timestamp('2015-10-19 00:00:00'), Timestamp('2015-10-20 00:00:00'), Timestamp('2015-10-21 00:00:00'), Timestamp('2015-10-22 00:00:00')]
dataset index head: [Timestamp('2015-10-16 00:00:00'), Timestamp('2015-10-19 00:00:00'), Timestamp('2015-10-20 00:00:00'), Timestamp('2015-10-21 00:00:00'), Timestamp('2015-10-22 00:00:00')]
indexes equal: True
target_return non-null in dataset: 2781


In [21]:
print("n splits:", len(splits))
if len(splits) > 0:
    train_pos, test_pos = splits[0]
    print("first train size:", len(train_pos), "first test size:", len(test_pos))
    print("first train range:", train_pos[0], "to", train_pos[-1])
    print("first test range:", test_pos[0], "to", test_pos[-1])
    print("last split test range:", splits[-1][1][0], "to", splits[-1][1][-1])
else:
    print("SPLITS IS EMPTY — this is the bug")

n splits: 29
first train size: 1000 first test size: 63
first train range: 0 to 999
first test range: 1000 to 1062
last split test range: 2764 to 2780
